In [ ]:
import os
import glob
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image


class IDCDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row["label"], dtype=torch.long)


def get_dataloaders(data_dir, batch_size=64, test_size=0.2):
    all_paths = glob.glob(os.path.join(data_dir, "**", "*.png"), recursive=True)

    data_list = []
    for path in all_paths:
        filename = os.path.basename(path)
        data_list.append({
            "path": path,
            "patient_id": filename.split("_")[0],
            "label": 1 if "class1" in filename else 0
        })

    df = pd.DataFrame(data_list)
    print(f"🕵️ 报告：在这个路径下，一共找到了 {len(df)} 张图片！")
    if df.empty:
        raise ValueError("一张图片都没找到！请检查数据路径是否正确。")

    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df["patient_id"]))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.Resize((50, 50)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    test_transform = transforms.Compose([
        transforms.Resize((50, 50)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    train_loader = DataLoader(
        IDCDataset(train_df, transform=train_transform),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    test_loader = DataLoader(
        IDCDataset(test_df, transform=test_transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return {"train": train_loader, "test": test_loader}


if __name__ == "__main__":
    TEST_DATA_DIR = r"C:\Users\Admin\Downloads\IDCai\data\archive"

    print("开始测试数据加载流水线...")
    try:
        loaders = get_dataloaders(TEST_DATA_DIR, batch_size=32)
        train_loader = loaders["train"]

        for images, labels in train_loader:
            print("\n✅ 数据流测试通过！")
            print(f"-> 批次图像张量形状: {images.shape}")
            print(f"-> 批次标签张量形状: {labels.shape}")
            break

    except FileNotFoundError:
        print("\n❌ 找不到图片，请检查 TEST_DATA_DIR 路径是否正确填写。")